# 🇮🇳 AI NSE Trading Research System
### Production-grade ML pipeline — RESEARCH USE ONLY

⚠️ **DISCLAIMER**: Does NOT guarantee profits. Significant financial risk involved.


In [ ]:
# STEP 0 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE_DIR = "/content/drive/MyDrive/ai-nse-trading"
for d in ["models","logs","data"]: os.makedirs(f"{DRIVE_DIR}/{d}", exist_ok=True)
print("Drive mounted.")

In [ ]:
# STEP 1 — Clone repo + install
REPO_URL = "https://github.com/manoj23798/ai-nse-trading.git"
REPO_DIR = "/content/ai-nse-trading"
import os, sys
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")
os.chdir(REPO_DIR)
# Handle nested folder structure
if os.path.exists(os.path.join(REPO_DIR, "ai-nse-trading")):
    REPO_DIR = os.path.join(REPO_DIR, "ai-nse-trading")
os.chdir(REPO_DIR)
os.system("pip install -q -r requirements.txt")
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("Ready. CWD:", os.getcwd())

In [ ]:
# STEP 2 — Configuration
TICKERS = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "THANGAMAYL.NS"]
USE_INTRADAY = False
DAILY_PERIOD = "5y"
WINDOW = 30
USE_TRANSFORMER = False  # True = more power, slightly slower
CAPITAL = 100_000
TRAIN_CONFIG = {"lr":1e-3,"batch_size":64,"epochs":60,"patience":10,"grad_clip":1.0,"weight_decay":1e-4}
print("Config ready. Tickers:", TICKERS)

In [ ]:
# STEP 3 — Download NSE Data
import os, sys
# Ensure src is in path
src_path = os.path.join(REPO_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import matplotlib; matplotlib.rcParams.update({"figure.facecolor":"#1a1a2e","axes.facecolor":"#16213e","text.color":"white","axes.labelcolor":"white","xtick.color":"white","ytick.color":"white"})
from data_loader import download_daily, download_intraday, time_split
raw_df = download_intraday(TICKERS, interval="15m", period="60d") if USE_INTRADAY else download_daily(TICKERS, period=DAILY_PERIOD)
print(f"Data shape: {raw_df.shape} | Range: {raw_df.date.min()} -> {raw_df.date.max()}")
raw_df.head()

In [ ]:
# STEP 4 — Feature Engineering
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from features import build_features, FEATURE_COLS, fit_and_scale, make_sequences
feat_df = build_features(raw_df, intraday=USE_INTRADAY)
print(f"Features: {feat_df.shape} | Cols: {len(FEATURE_COLS)}")
feat_df[FEATURE_COLS].describe().round(3)

In [ ]:
# STEP 5 — Visualise Price + Indicators
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from evaluate import plot_price_with_indicators
PLOT_TIC = TICKERS[0]
plot_price_with_indicators(feat_df[feat_df.tic==PLOT_TIC].tail(120).copy(), PLOT_TIC, save_path=f"logs/indicators_{PLOT_TIC}.png")

In [ ]:
# STEP 6 — Train Direction Models (all tickers)
import numpy as np, os, shutil, sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from models import build_model, load_model, DEVICE
from train import train_model, save_to_drive
from evaluate import plot_training_history
print(f"Device: {DEVICE}")
trained_models = {}
for TIC in TICKERS:
    print(f"\n=== {TIC} ===")
    tic_df = feat_df[feat_df.tic==TIC].copy()
    if len(tic_df) < WINDOW+50: print(f"Skip {TIC}"); continue
    train_df, val_df, test_df = time_split(tic_df)
    scaler_path = f"models/scaler_{TIC}.pkl"
    X_train, X_val, X_test, scaler = fit_and_scale(train_df, val_df, test_df, FEATURE_COLS, method="standard", save_path=scaler_path)
    y_train = train_df.target_direction.values.astype("float32")
    y_val   = val_df.target_direction.values.astype("float32")
    y_test  = test_df.target_direction.values.astype("float32")
    X_tr_seq,y_tr_seq = make_sequences(X_train,y_train,window=WINDOW)
    X_vl_seq,y_vl_seq = make_sequences(X_val,  y_val,  window=WINDOW)
    X_te_seq,y_te_seq = make_sequences(X_test, y_test, window=WINDOW)
    mtype = "transformer" if USE_TRANSFORMER else "lstm_direction"
    model = build_model(mtype, input_size=len(FEATURE_COLS))
    local_ckpt = f"models/{mtype}_{TIC}.pt"
    drive_ckpt = f"{DRIVE_DIR}/models/{mtype}_{TIC}.pt"
    if os.path.exists(drive_ckpt): shutil.copy2(drive_ckpt,local_ckpt); model=load_model(model,local_ckpt); print("  Loaded from Drive.")
    history = train_model(model,X_tr_seq,y_tr_seq,X_vl_seq,y_vl_seq,task="classification",config=TRAIN_CONFIG,save_path=local_ckpt)
    try: save_to_drive(local_ckpt, DRIVE_DIR+"/models"); save_to_drive(scaler_path, DRIVE_DIR+"/models")
    except Exception as e: print(f"Drive save: {e}")
    trained_models[TIC] = {"model":model,"scaler":scaler,"X_test":X_te_seq,"y_test":y_te_seq,"test_df":test_df,"history":history}
    plot_training_history(history, save_path=f"logs/training_{TIC}.png")
print("All models trained!")

In [ ]:
# STEP 7 — Evaluate on Test Set
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from evaluate import predict_classification, eval_classification, plot_predicted_vs_actual, log_predictions
for TIC, bundle in trained_models.items():
    print(f"\n--- {TIC} ---")
    probs = predict_classification(bundle["model"], bundle["X_test"])
    eval_classification(bundle["y_test"], probs)
    dates = bundle["test_df"]["date"].values[WINDOW:]
    log_predictions(dates, bundle["y_test"], probs, tic=TIC, label="direction")
    plot_predicted_vs_actual(dates, bundle["y_test"], probs, title=f"{TIC} — Prob vs Actual", save_path=f"logs/pred_vs_actual_{TIC}.png")

In [ ]:
# STEP 8 — Train High/Low Model
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from sklearn.preprocessing import MinMaxScaler
hl_models = {}
for TIC in TICKERS:
    if TIC not in trained_models: continue
    print(f"\nHigh/Low model: {TIC}")
    tic_df = feat_df[feat_df.tic==TIC].copy()
    train_df,val_df,test_df = time_split(tic_df)
    sc = trained_models[TIC]["scaler"]
    X_train,X_val,X_test = sc.transform(train_df[FEATURE_COLS].values), sc.transform(val_df[FEATURE_COLS].values), sc.transform(test_df[FEATURE_COLS].values)
    hl_sc = MinMaxScaler()
    y_tr_hl = hl_sc.fit_transform(train_df[["target_high","target_low"]].values)
    y_vl_hl = hl_sc.transform(val_df[["target_high","target_low"]].values)
    y_te_hl = hl_sc.transform(test_df[["target_high","target_low"]].values)
    X_tr_seq,y_tr_hl2 = make_sequences(X_train,y_tr_hl,window=WINDOW)
    X_vl_seq,y_vl_hl2 = make_sequences(X_val,y_vl_hl,window=WINDOW)
    X_te_seq,y_te_hl2 = make_sequences(X_test,y_te_hl,window=WINDOW)
    hl_model = build_model("high_low", input_size=len(FEATURE_COLS))
    hl_path = f"models/high_low_{TIC}.pt"
    train_model(hl_model,X_tr_seq,y_tr_hl2,X_vl_seq,y_vl_hl2,task="regression_hl",config={**TRAIN_CONFIG,"epochs":40},save_path=hl_path)
    try: save_to_drive(hl_path, DRIVE_DIR+"/models")
    except: pass
    hl_models[TIC] = {"model":hl_model,"scaler":hl_sc}
print("High/Low models done!")

In [ ]:
# STEP 9 — Signal Generation + Risk Management
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from strategy import generate_signals, RISK_CONFIG
signal_dfs = {}
for TIC, bundle in trained_models.items():
    probs = predict_classification(bundle["model"], bundle["X_test"])
    test_aligned = bundle["test_df"].iloc[WINDOW:].copy().reset_index(drop=True)
    sig_df = generate_signals(test_aligned, probs, config={**RISK_CONFIG,"capital":CAPITAL})
    signal_dfs[TIC] = sig_df
    print(f"\n{TIC} signals:")
    print(sig_df[["date","close","signal","prob_up","stop_loss","take_profit"]].tail(5))

In [ ]:
# STEP 10 — Backtesting
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from backtest import run_backtest, walk_forward_backtest, plot_equity_curve
backtest_results = {}
for TIC, sig_df in signal_dfs.items():
    print(f"\nBacktest: {TIC}")
    result = run_backtest(sig_df, capital=CAPITAL)
    backtest_results[TIC] = result
    plot_equity_curve(result, title=f"{TIC} Equity Curve", save_path=f"logs/equity_{TIC}.png")

In [ ]:
# STEP 11 — Summary + Save to Drive
import pandas as pd, shutil, sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
rows = [{**r["metrics"], "ticker":t} for t,r in backtest_results.items()]
summary = pd.DataFrame(rows).set_index("ticker")
print("\n📊 BACKTEST SUMMARY\n", summary.to_string())
summary.to_csv(f"{DRIVE_DIR}/backtest_summary.csv")
try: shutil.copytree("logs", DRIVE_DIR+"/logs", dirs_exist_ok=True); print("Logs → Drive")
except Exception as e: print(e)

In [ ]:
# STEP 12 — Daily Continuous Learning Loop
# Run this cell EVERY DAY to update models with new market data
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from continuous_learning import run_daily_loop
run_daily_loop(
    tickers=TICKERS,
    model_type="lstm_direction",
    retrain_on_last_n_days=90,
    drive_dir=DRIVE_DIR,
    use_intraday=USE_INTRADAY,
)